# Module 06 — Lesson 03
# Detecting and Handling Outliers (IQR, Z-Score)

---

> Picking up the same employee data after Lessons 01 and 02 (missing values handled, duplicates resolved), one more problem hides in the numbers: a handful of values that don't look like the rest. Some are typos. One is a real person. Telling them apart — and picking a detection method that doesn't fool itself — is what this lesson is about.

---
## 🎯 Learning Objectives

By the end of this lesson, you will be able to:

- Detect outliers with the **IQR method** (quartiles and the 1.5×IQR fence)
- Detect outliers with the **Z-score method**, and explain why it can be fooled by the very outliers it's trying to find (the *masking effect*)
- Use a **robust (median/MAD-based) Z-score** as a fix for that weakness
- Investigate *why* a value is extreme before deciding what to do with it — a typo, a data glitch, or a genuine real-world extreme value each need a different fix
- Apply the right fix per case: correct, cap (winsorize), remove, or keep unchanged

In [1]:
import pandas as pd

employees = pd.read_csv("data/employees_clean.csv")
print(f"employees.shape: {employees.shape}")
employees[["employee_id", "name", "department", "age", "salary"]].describe()

employees.shape: (49, 7)


,employee_id,age,salary
count,49.000000,49.000000,49.00000
mean,1030.061224,37.265306,84038.77551
std,18.201887,10.735020,102407.39682
min,1001.000000,5.000000,-73200.00000
25%,1015.000000,28.000000,57200.00000
50%,1028.000000,38.000000,70700.00000
75%,1046.000000,43.000000,81800.00000
max,1060.000000,57.000000,744000.00000


The `describe()` table is already a clue: look at `salary`'s max (744,000) against its 75th percentile (81,800), and its min (-73,200) against a minimum wage that should never be negative. `age`'s minimum of 5 is impossible for an employee. Something is off in more than one row — let's find exactly which ones, systematically.

---
## 1. The IQR Method

The **interquartile range (IQR)** is the spread of the middle 50% of the data: `Q3 - Q1`. Anything further than `1.5 × IQR` below Q1 or above Q3 is flagged as an outlier. It's a **robust** method — quartiles don't move much even if a few extreme values are added, because they're based on rank (position in sorted order), not on the actual magnitude of every value.

In [2]:
def iqr_outliers(series):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return lower, upper, (series < lower) | (series > upper)

age_lower, age_upper, age_outlier_mask = iqr_outliers(employees["age"])
print(f"age: normal range is [{age_lower}, {age_upper}]")
employees[age_outlier_mask][["employee_id", "name", "age"]]

age: normal range is [5.5, 65.5]


,employee_id,name,age
18,1021,Vikram Pillai,5.0


In [3]:
salary_lower, salary_upper, salary_outlier_mask = iqr_outliers(employees["salary"])
print(f"salary: normal range is [{salary_lower:,.0f}, {salary_upper:,.0f}]")
employees[salary_outlier_mask][["employee_id", "name", "department", "salary"]]

salary: normal range is [20,300, 118,700]


,employee_id,name,department,salary
2,1003,Karan Kapoor,Engineering,744000.0
28,1034,Preeti Patel,Engineering,245000.0
37,1047,Simran Patel,Engineering,-73200.0


IQR flags all three unusual salaries — the two typos *and* the one genuine high earner — because it doesn't know or care *why* a value is extreme, only that it is. That's expected; IQR is a detector, not a judge. We'll decide what to do with each one in Section 4.

---
## 2. The Z-Score Method — and Where It Breaks

The classic **Z-score** measures how many standard deviations a value sits from the mean: `z = (x - mean) / std`. A common rule of thumb flags anything with `|z| > 3`. Let's run it on `salary` and see what it catches.

In [4]:
salary_mean = employees["salary"].mean()
salary_std = employees["salary"].std()
z_scores = (employees["salary"] - salary_mean) / salary_std

print(f"salary mean: {salary_mean:,.0f}, salary std: {salary_std:,.0f}")
print()
print("Rows flagged with |z| > 3:")
employees.assign(z_score=z_scores.round(2))[abs(z_scores) > 3][["employee_id", "name", "salary", "z_score"]]

salary mean: 84,039, salary std: 102,407

Rows flagged with |z| > 3:


,employee_id,name,salary,z_score
2,1003,Karan Kapoor,744000.0,6.44


Only **one** row got flagged. But Section 1 found three unusual salaries — where did the other two go? Check their z-scores directly.

In [5]:
check_ids = [1034, 1047]  # the legitimate high earner, and the negative-salary typo
employees.assign(z_score=z_scores.round(2))[employees["employee_id"].isin(check_ids)][
    ["employee_id", "name", "salary", "z_score"]
]

,employee_id,name,salary,z_score
28,1034,Preeti Patel,245000.0,1.57
37,1047,Simran Patel,-73200.0,-1.54


Neither one crosses `|z| > 3` — the -73,200 typo scores only about -1.5. This is the **masking effect**: the standard deviation itself is computed from the *same* data that includes the 744,000 typo. That one extreme value inflates `std` so much that it drags the "normal" range wide enough to swallow the other two outliers. The very thing you're trying to detect distorts the ruler you're measuring it with.

---
## 3. A Fix: the Robust (Median / MAD) Z-Score

Swap the mean and standard deviation — both sensitive to extreme values — for the **median** and the **median absolute deviation (MAD)**, both rank-based and much harder to distort. This is sometimes called the *modified Z-score*.

In [6]:
def robust_z_scores(series):
    median = series.median()
    mad = (series - median).abs().median()
    return 0.6745 * (series - median) / mad  # 0.6745 rescales MAD to match std under a normal distribution

robust_z = robust_z_scores(employees["salary"])
print("Rows flagged with |modified z-score| > 3.5 (the standard threshold for this method):")
employees.assign(robust_z=robust_z.round(2))[abs(robust_z) > 3.5][
    ["employee_id", "name", "salary", "robust_z"]
].sort_values("robust_z", ascending=False)

Rows flagged with |modified z-score| > 3.5 (the standard threshold for this method):


,employee_id,name,salary,robust_z
2,1003,Karan Kapoor,744000.0,38.16
28,1034,Preeti Patel,245000.0,9.88
37,1047,Simran Patel,-73200.0,-8.16


All three flagged again, matching the IQR result. **Takeaway: prefer IQR or a robust Z-score over the standard mean/std Z-score whenever a dataset might already contain more than one outlier** — which, realistically, is almost always.

---
## 4. Investigate Before You Fix — Not Every Outlier Gets the Same Treatment

Detection found four unusual values. None of them get the same fix, because they don't have the same cause:

| Row | Value | Likely cause | Fix |
|---|---|---|---|
| `1021` age = 5 | Impossible for an employee | Data-entry typo (missing a digit — probably 50) | Correct it, or treat as missing and re-impute (Lesson 01's toolkit) |
| `1003` salary = 744,000 | 10× every peer in the department | Data-entry typo (an extra digit) | Correct it — divide by 10, or treat as missing and re-impute |
| `1047` salary = -73,200 | Salary cannot be negative | Data glitch (sign flipped on import) | Correct it — take the absolute value |
| `1034` salary = 245,000 | High, but department/experience explain it | **Genuine value** — a senior Engineering employee with 22 years of experience | Keep it — this is real information, not an error |

The rule that separates row 1034 from the other three: **does the rest of the row explain the extreme value?** A 22-year veteran senior engineer earning far more than a 2-year junior is expected, not broken. An employee whose age is 5 has no such explanation anywhere in the row.

In [7]:
employees_fixed = employees.copy()

# Typo: age missing a digit (5 -> 50, the plausible intended value)
employees_fixed.loc[employees_fixed["employee_id"] == 1021, "age"] = 50

# Typo: an extra digit inflated the salary 10x
employees_fixed.loc[employees_fixed["employee_id"] == 1003, "salary"] = (
    employees_fixed.loc[employees_fixed["employee_id"] == 1003, "salary"] / 10
)

# Glitch: sign flipped during import
employees_fixed.loc[employees_fixed["employee_id"] == 1047, "salary"] = (
    employees_fixed.loc[employees_fixed["employee_id"] == 1047, "salary"].abs()
)

# 1034 (Preeti Patel, 245,000) is left untouched -- it's real

print("Before vs after, for the four rows we investigated:")
fixed_ids = [1021, 1003, 1047, 1034]
comparison = employees[employees["employee_id"].isin(fixed_ids)][["employee_id", "age", "salary"]].merge(
    employees_fixed[employees_fixed["employee_id"].isin(fixed_ids)][["employee_id", "age", "salary"]],
    on="employee_id", suffixes=("_before", "_after")
)
comparison

Before vs after, for the four rows we investigated:


,employee_id,age_before,salary_before,age_after,salary_after
0,1003,25.0,744000.0,25.0,74400.0
1,1021,5.0,51100.0,50.0,51100.0
2,1034,54.0,245000.0,54.0,245000.0
3,1047,27.0,-73200.0,27.0,73200.0


In [8]:
# Re-run IQR detection to confirm the dataset is genuinely clean now (aside from the real outlier)
_, _, age_outliers_after = iqr_outliers(employees_fixed["age"])
_, _, salary_outliers_after = iqr_outliers(employees_fixed["salary"])

print(f"Age outliers remaining: {age_outliers_after.sum()}")
print(f"Salary outliers remaining: {salary_outliers_after.sum()} "
      f"({employees_fixed[salary_outliers_after]['name'].tolist()} -- expected, it's genuine)")

Age outliers remaining: 0
Salary outliers remaining: 1 (['Preeti Patel'] -- expected, it's genuine)


---
## ⚠️ Common Mistakes

In [9]:
# -- Mistake 1: Using the standard mean/std Z-score as the ONLY check ------------------------

print("Mistake 1 — trusting mean/std Z-score alone would have missed 2 of the 3 unusual")
print("salaries, because the 744,000 typo inflated the standard deviation and masked the rest:")
print(f"  Outliers found by standard Z-score (|z| > 3): {(abs(z_scores) > 3).sum()}")
print(f"  Outliers found by IQR: {salary_outlier_mask.sum()}")
print("  -> Prefer IQR or a robust (median/MAD) Z-score when more than one outlier may exist.")

Mistake 1 — trusting mean/std Z-score alone would have missed 2 of the 3 unusual
salaries, because the 744,000 typo inflated the standard deviation and masked the rest:
  Outliers found by standard Z-score (|z| > 3): 1
  Outliers found by IQR: 3
  -> Prefer IQR or a robust (median/MAD) Z-score when more than one outlier may exist.


In [10]:
# -- Mistake 2: Deleting every flagged row without checking WHY it's extreme -----------------

naive_drop = employees[~salary_outlier_mask]
print("Mistake 2 — blindly dropping every IQR-flagged salary would also delete Preeti Patel's")
print("row (employee_id 1034) -- a real, correctly recorded senior employee's salary, not an error:")
print(f"  Rows before naive drop: {employees.shape[0]}")
print(f"  Rows after naive drop:  {naive_drop.shape[0]}  (lost a real data point, not just noise)")
print("  -> Investigate each flagged row before deciding to fix, cap, or remove it.")

Mistake 2 — blindly dropping every IQR-flagged salary would also delete Preeti Patel's
row (employee_id 1034) -- a real, correctly recorded senior employee's salary, not an error:
  Rows before naive drop: 49
  Rows after naive drop:  46  (lost a real data point, not just noise)
  -> Investigate each flagged row before deciding to fix, cap, or remove it.


In [11]:
# -- Mistake 3: Applying the SAME fix (e.g. always cap at the IQR fence) to every outlier -----

capped_naively = employees["salary"].clip(lower=salary_lower, upper=salary_upper)
print("Mistake 3 — capping every flagged salary at the IQR fence 'fixes' the 744,000 typo")
print("reasonably, but also silently caps the legitimate 245,000 salary down to the fence value,")
print("erasing a true difference in seniority and pay:")
print(f"  744,000 typo capped to: {capped_naively[employees['employee_id'] == 1003].iloc[0]:,.0f}")
print(f"  245,000 REAL salary capped to: {capped_naively[employees['employee_id'] == 1034].iloc[0]:,.0f}")
print("  -> Capping is a reasonable default ONLY for values you can't otherwise correct or explain.")

Mistake 3 — capping every flagged salary at the IQR fence 'fixes' the 744,000 typo
reasonably, but also silently caps the legitimate 245,000 salary down to the fence value,
erasing a true difference in seniority and pay:
  744,000 typo capped to: 118,700
  245,000 REAL salary capped to: 118,700
  -> Capping is a reasonable default ONLY for values you can't otherwise correct or explain.


---
## ✏️ Practice Exercises

### Exercise 1 — Starter: IQR on `years_experience`

Load `data/employees_clean.csv` fresh. Use the `iqr_outliers()` pattern from Section 1 to check `years_experience` for outliers. Print the normal range and any flagged rows.

In [12]:
# Your code here

### Exercise 2 — Compare Both Z-Score Methods on `age`

Compute both the standard Z-score and the robust (median/MAD) Z-score for the `age` column. Do they agree on which row(s) are outliers? Print both sets of flagged rows.

In [13]:
# Your code here

### Exercise 3 — Investigate a Flag

For every row flagged as a salary outlier by IQR, print the row's full data (department, age, years_experience, salary). For each one, write a one-line comment: does the rest of the row explain the extreme salary, or does it look like an error?

In [14]:
# Your code here

### Exercise 4 — Fix Only What's Broken

Starting from the raw `employees_clean.csv`, correct the `age` typo and the two salary errors (the 10x typo and the negative sign), but leave the genuine high earner untouched. Confirm with `iqr_outliers()` that only the real outlier remains flagged afterward.

In [15]:
# Your code here

### Exercise 5 — (Challenge) Winsorize Instead of Correcting

Sometimes you can't tell whether a flagged value is an error or genuine — there's no metadata to check against. Write a function `winsorize(series)` that caps values at the IQR fences (instead of dropping or manually correcting them) and apply it to a copy of the *uncorrected* `salary` column. Compare the mean salary before and after.

In [16]:
# Your code here

---
## 📌 Key Takeaways

- **IQR (quartile-based) outlier detection is robust to multiple outliers; the standard mean/std Z-score is not** — one extreme value can inflate the standard deviation enough to mask others (the masking effect). Prefer IQR or a median/MAD-based robust Z-score by default.
- **An outlier is a statistical fact, not a verdict** — always check whether the rest of the row explains the extreme value before deciding it's an error.
- **The fix depends on the cause: correct typos, take the absolute value for sign glitches, cap (winsorize) when you can't otherwise explain or correct a value, and leave genuine extreme values alone** — a single blanket rule ("drop everything flagged" or "cap everything flagged") will damage real data as often as it fixes bad data.